In [1]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_chroma import Chroma
from  langchain_openai import ChatOpenAI, OpenAIEmbeddings

from dotenv import load_dotenv

In [2]:
load_dotenv()

def partition_documents(file_path):
    elements=partition_pdf(
        filename=file_path,
        strategy='hi_res',
        infer_table_structure=True,
        extract_image_block_types=['Image'],
        extract_image_block_to_payload=True
    )
    
    return elements

file_path='/home/obs/Desktop/RAG/RAG_initial/notebooks/doc/attention-is-all-you-need-1.pdf'
elements=partition_documents(file_path)

No languages specified, defaulting to English.


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

In [3]:
def create_chunks_by_title(elements):
    chunks=chunk_by_title(
        elements=elements,
        max_characters=3000,
        new_after_n_chars=2500,
        combine_text_under_n_chars=500
    )
    return chunks

chunks=create_chunks_by_title(elements)

In [ ]:
def seperate_content(chunk):
    content_data={
        "text":chunk.text,
        "tables":[],
        "images":[],
        "type":['text']
    }
    if hasattr(chunk, 'metadata') and hasattr(chunk.metedata, "orgi_elements"):
        for element in chunk:
            element_type=type(element).__name__
            
            if element_type == 'Table':
                content_data['type'].append('table')
                table_as_html=getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_as_html)
            
            elif element_type == 'Image':
                content_data['type'].append('image')
                image_base64=getattr(element.metadata, 'image_base64', element.text)
                content_data['images'].append(image_base64)
    content_data['type']=list(set(content_data['type']))
    return content_data    

def  create_ai_summary(chunks):
    print('createing Ai summary')
    
    langchain_documents=[]
    length=len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk=i+1
        print(f'\n current processingchunk is {current_chunk}/{length}')
        
        seperate_content(chunk)